In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain_core.chatopenai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableSequence
import os
from dotenv import load_dotenv


load_dotenv()



# generation = input("Enter your generation (e.g., GenZ, Millennial, GenX): ")

True

### ***Chains with Custom Runnables***

##### Chain 1

In [24]:
# Task 1 - Prompt Template Creation
user_input = input("Enter a movie name: ")
prompt_template = ChatPromptTemplate.from_messages([("user", "{topic}"),
("system", "You are a movie summary generator who answers in a concise and witty manner.")])

In [25]:
# Task 2 - Prompt Finalization.

ready_prompt = prompt_template.invoke({"topic": user_input, "generation": "millennial"})
llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.7)

In [26]:
# Task 3 - String Output Parser Creation

str_parse = StrOutputParser()

In [27]:
def dictionary_parser(text:str)->dict:

    return {"text": text}

In [28]:
prompt_post = ChatPromptTemplate.from_messages([("user", "Create a movie review from the summary {text}"),
("system", "You are a movie critic on Twitter.")])

In [29]:
linkedin_chain =  dictionary_parser | prompt_post | llm_openai | str_parse

##### Chain 2

In [30]:
def insta_chain(text):

    # print(text)

    text = text
    #Task 1 - Prompt Template Creation
    insta_prompt = ChatPromptTemplate.from_messages([("user", "Create an Instagram movie review from the summary {text}"),
    ("system", "You are a social media influencer. You need to post on Instagram based on the text that is provided to you.")])

    llm_openai= init_chat_model(model=os.environ.get("LLM_MODEL"), temperature = 0.7)

    str_parse = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parse

    result = chain_insta.invoke({"text": text})

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

#### ***Orchestrator***

In [31]:
# Chain with custom task

final_chain = (prompt_template | llm_openai | str_parse |
RunnableParallel({"linkedin": linkedin_chain, "instagram": insta_chain_runnable}))

final_chain.invoke({"topic": user_input})

{'linkedin': 'Fighter roars into the skies: Hrithik Roshan anchors a glossy, high‑altitude Bollywood actioner that serves thunderous dogfights, gleaming VFX and a training‑montage heart. Rivalry, loss and duty forge a tight squadron — big on spectacle, short on subtlety. Watch for the aerial choreography; skip if you crave nuance. ★★★/5 #Fighter',
 'instagram': 'FIGHTER ✈️🔥 Hrithik Roshan goes full throttle in this glossy, high‑altitude Bollywood actioner — think thunderous dogfights, training‑montage bonding, and a revenge arc wrapped in patriotic stakes. Big on spectacle, light on subtlety.  \n\nWhat you get: jaw‑dropping aerial set pieces, a tight squadron chemistry, and heart under all that engine noise. What you don’t get: nuanced quiet moments. If you want sky‑diving cinema with heart and LOUD engines, this is your kind of ride.  \n\nRating: 4/5 — pure blockbuster fuel.  \n\nSeen it yet? Tell me your favorite dogfight moment 👇  \n#Fighter #HrithikRoshan #Bollywood #ActionMovie #S